# 单个 TSFM 的细粒度误差分析（增强版）

本 notebook 聚焦 **单个 TSFM**，并在全部窗口上做如下分析：
1. 使用 **sMAPE** 作为统一误差指标；
2. 自动支持中文字体，完善图题与上下文信息（TSFM / 指标 / 数据集范围）；
3. 对每一个历史特征分别绘制“特征分桶 -> 误差统计”图；
4. 做误差数学分析：自动搜索一个**最大化桶间误差差异**的特征分桶方案。


In [ ]:
import os
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display
from matplotlib import font_manager

from analysis.tsfm_error_analysis import (
    list_tsfms,
    load_tsfm_window_dataframe,
    add_quantile_buckets,
    compute_feature_combo_error_stats,
    build_error_interval_feature_stats,
    plot_feature_pair_heatmaps,
    plot_error_bucket_feature_heatmap,
)

# ========= 中文字体自动配置 =========
def setup_chinese_font():
    preferred = [
        'Noto Sans CJK SC', 'Noto Sans CJK', 'Microsoft YaHei',
        'SimHei', 'PingFang SC', 'Heiti SC', 'WenQuanYi Zen Hei',
        'Arial Unicode MS'
    ]
    installed = {f.name for f in font_manager.fontManager.ttflist}
    chosen = None
    for name in preferred:
        if name in installed:
            chosen = name
            break
    if chosen is not None:
        plt.rcParams['font.sans-serif'] = [chosen, 'DejaVu Sans']
    else:
        plt.rcParams['font.sans-serif'] = ['DejaVu Sans']
    plt.rcParams['axes.unicode_minus'] = False
    return chosen

CH_FONT = setup_chinese_font()
print('已启用字体:', CH_FONT if CH_FONT else '未找到中文字体，使用默认 DejaVu Sans')

sns.set_theme(style='whitegrid', context='talk')


In [ ]:
# ====== 配置区 ======
OUTPUT_ROOT = 'correction_datasets_double_res_0'
TSFM_NAME = None  # 例如 'chronos_bolt_tiny'；None 时自动取第一个

# 数据范围过滤（None / [] 表示不过滤）
DATASET_FILTERS = None      # 示例: ['qb_dataset_a', 'qb_dataset_b']
FREQ_FILTERS = None         # 示例: ['daily']
DOMAIN_FILTERS = None       # 示例: ['finance']

# 分桶参数
FEATURE_BINS = 5
ERROR_BINS = 6
ERROR_METRIC = 'smape'      # 按你的要求，固定使用 sMAPE
MIN_BUCKET_COUNT = 30       # 数学分析时的桶最小样本数约束

all_tsfms = list_tsfms(OUTPUT_ROOT)
if not all_tsfms:
    raise RuntimeError(f'未在 {OUTPUT_ROOT} 找到 TSFM 目录')
if TSFM_NAME is None:
    TSFM_NAME = all_tsfms[0]
print('可用 TSFM:', all_tsfms)
print('当前 TSFM:', TSFM_NAME)


In [ ]:
# ====== 载入并过滤数据 ======
df = load_tsfm_window_dataframe(OUTPUT_ROOT, TSFM_NAME)
if df.empty:
    raise RuntimeError(f'未加载到数据: {TSFM_NAME}')

if DATASET_FILTERS:
    df = df[df['dataset'].isin(DATASET_FILTERS)].copy()
if FREQ_FILTERS:
    df = df[df['freq'].isin(FREQ_FILTERS)].copy()
if DOMAIN_FILTERS:
    df = df[df['domain'].isin(DOMAIN_FILTERS)].copy()

if df.empty:
    raise RuntimeError('过滤后数据为空，请检查 DATASET_FILTERS/FREQ_FILTERS/DOMAIN_FILTERS')

scope_datasets = sorted(df['dataset'].unique().tolist())
scope_text = f"datasets={len(scope_datasets)} | samples={len(df)}"

print('样本数:', len(df))
print('数据集范围:', scope_datasets[:8], '...' if len(scope_datasets) > 8 else '')
print('频率范围:', sorted(df['freq'].astype(str).unique().tolist()))
print('领域范围:', sorted(df['domain'].astype(str).unique().tolist()))
display(df.head(3))


In [ ]:
# ====== 特征列 ======
default_feature_cols = [
    'hist_mean', 'hist_std', 'hist_range_p10_p90',
    'trend_slope', 'trend_strength', 'remainder_strength',
    'fft_low_ratio', 'fft_high_ratio', 'fft_dominant_freq',
]
feature_cols = [c for c in default_feature_cols if c in df.columns]
print('可用特征:', feature_cols)

# 预分桶（后续多个单元复用）
df_b = add_quantile_buckets(df, feature_cols=feature_cols, n_bins=FEATURE_BINS)


In [ ]:
# ====== 全局误差可视化（sMAPE） ======
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

sns.histplot(df[ERROR_METRIC], bins=50, kde=True, ax=axes[0], color='#4C72B0')
axes[0].set_title(f"{TSFM_NAME} | 全局{ERROR_METRIC.upper()}分布 | {scope_text}")
axes[0].set_xlabel(ERROR_METRIC.upper())

# 仅展示样本量最大的前10个数据集
top_ds = df['dataset'].value_counts().head(10).index
box_df = df[df['dataset'].isin(top_ds)].copy()
order = box_df.groupby('dataset')[ERROR_METRIC].mean().sort_values(ascending=False).index
sns.boxplot(data=box_df, x='dataset', y=ERROR_METRIC, order=order, ax=axes[1], color='#55A868')
axes[1].set_title(f"{TSFM_NAME} | Top-10数据集{ERROR_METRIC.upper()}箱线图 | {scope_text}")
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


In [ ]:
# ====== 每个特征各画一张：特征分桶 -> sMAPE统计 ======
def plot_feature_smape_profile(df_bucketed, feature, metric='smape'):
    bucket_col = f'{feature}_bucket'
    g = (
        df_bucketed.groupby(bucket_col, observed=True)[metric]
        .agg(['mean', 'max', 'std', 'count'])
        .reset_index()
    )
    g['bucket_str'] = g[bucket_col].astype(str)

    fig, ax1 = plt.subplots(figsize=(12, 5))
    x = np.arange(len(g))

    bars = ax1.bar(x, g['mean'], color='#4C72B0', alpha=0.85, label=f'{metric.upper()} Mean')
    ax1.plot(x, g['max'], color='#C44E52', marker='o', label=f'{metric.upper()} Max')
    ax1.fill_between(x, g['mean'] - g['std'].fillna(0), g['mean'] + g['std'].fillna(0),
                     color='#8172B2', alpha=0.2, label=f'{metric.upper()} ±1STD')

    ax1.set_xticks(x)
    ax1.set_xticklabels(g['bucket_str'], rotation=35, ha='right')
    ax1.set_xlabel(f'{feature} quantile-bucket')
    ax1.set_ylabel(metric.upper())
    ax1.set_title(f"TSFM={TSFM_NAME} | 指标={metric.upper()} | 特征={feature} | {scope_text}")

    ax2 = ax1.twinx()
    ax2.plot(x, g['count'], color='#55A868', marker='s', linestyle='--', label='Count')
    ax2.set_ylabel('Count')

    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(handles1 + handles2, labels1 + labels2, loc='upper left')

    plt.tight_layout()
    plt.show()

for feat in feature_cols:
    plot_feature_smape_profile(df_b, feat, metric=ERROR_METRIC)


In [ ]:
# ====== 选取最相关的2个特征做双特征热力图（sMAPE均值/最大/标准差） ======
if len(feature_cols) >= 2:
    corr = df[feature_cols + [ERROR_METRIC]].corr(numeric_only=True)[ERROR_METRIC].drop(ERROR_METRIC)
    top2 = corr.abs().sort_values(ascending=False).head(2).index.tolist()
    stats = compute_feature_combo_error_stats(df_b, top2, error_metric=ERROR_METRIC)

    fig = plot_feature_pair_heatmaps(stats, f'{top2[0]}_bucket', f'{top2[1]}_bucket')
    fig.suptitle(
        f"TSFM={TSFM_NAME} | 指标={ERROR_METRIC.upper()} | 双特征桶热力图 ({top2[0]} × {top2[1]}) | {scope_text}",
        y=1.05,
    )
    plt.show()


In [ ]:
# ====== 按误差分桶后观察特征画像（sMAPE桶） ======
err_feat_stats = build_error_interval_feature_stats(
    df,
    error_metric=ERROR_METRIC,
    feature_cols=feature_cols,
    n_error_bins=ERROR_BINS,
)

display(err_feat_stats.head())
for stat in ['mean', 'max', 'std']:
    fig = plot_error_bucket_feature_heatmap(err_feat_stats, feature_cols, stat=stat, figsize=(14, 6))
    fig.suptitle(f"TSFM={TSFM_NAME} | 指标={ERROR_METRIC.upper()} | 误差桶内特征{stat} | {scope_text}", y=1.03)
    plt.show()


## 数学分析：寻找“桶间误差差异最大化”的分桶方案

目标：在基于特征的分桶中，寻找让各桶 **sMAPE 均值差异**最大的方案。

我们定义一个综合目标（越大越好）：

\[
	ext{score} = \eta^2 \cdot \log(1 + \Delta_{\max})
\]

其中：
- \(\eta^2 = rac{SS_{between}}{SS_{total}}\)：表示“桶解释了多少误差方差”；
- \(\Delta_{\max} = \max_b \mu_b - \min_b \mu_b\)：桶均值之间的最大差。

并加入约束：每个桶至少有 `MIN_BUCKET_COUNT` 个样本，避免极小桶造成虚高差异。


In [ ]:
# ====== 自动搜索最优分桶方案（单特征 + 双特征） ======
def evaluate_bucket_scheme(df_in, features, n_bins=5, metric='smape', min_count=30):
    tmp = df_in.copy()

    # 建立桶列
    bucket_cols = []
    for f in features:
        col = f'{f}_bucket_opt'
        tmp[col] = pd.qcut(tmp[f], q=n_bins, duplicates='drop')
        bucket_cols.append(col)

    # 组合桶标签
    combo_col = '_combo_bucket_'
    tmp[combo_col] = tmp[bucket_cols].astype(str).agg(' | '.join, axis=1)

    g = tmp.groupby(combo_col, observed=True)[metric].agg(['mean', 'count'])
    g = g[g['count'] >= min_count]
    if len(g) < 2:
        return None

    # 仅保留有效桶后重算eta^2
    valid_buckets = set(g.index.tolist())
    work = tmp[tmp[combo_col].isin(valid_buckets)].copy()
    overall_mean = work[metric].mean()

    # SS_total, SS_between
    ss_total = np.sum((work[metric] - overall_mean) ** 2)
    ss_between = 0.0
    for b, row in g.iterrows():
        ss_between += row['count'] * (row['mean'] - overall_mean) ** 2
    eta2 = float(ss_between / (ss_total + 1e-12))

    gap = float(g['mean'].max() - g['mean'].min())
    score = float(eta2 * np.log1p(gap))

    return {
        'features': tuple(features),
        'n_bins': int(n_bins),
        'num_valid_buckets': int(len(g)),
        'min_bucket_count': int(g['count'].min()),
        'max_bucket_count': int(g['count'].max()),
        'eta2': eta2,
        'mean_gap': gap,
        'score': score,
    }

results = []
# 单特征
for f in feature_cols:
    for b in [3, 4, 5, 6]:
        r = evaluate_bucket_scheme(df, [f], n_bins=b, metric=ERROR_METRIC, min_count=MIN_BUCKET_COUNT)
        if r is not None:
            results.append(r)

# 双特征（组合分桶）
for f1, f2 in itertools.combinations(feature_cols, 2):
    for b in [3, 4, 5]:
        r = evaluate_bucket_scheme(df, [f1, f2], n_bins=b, metric=ERROR_METRIC, min_count=MIN_BUCKET_COUNT)
        if r is not None:
            results.append(r)

res_df = pd.DataFrame(results).sort_values('score', ascending=False).reset_index(drop=True)
print('候选方案数:', len(res_df))
display(res_df.head(20))


In [ ]:
# ====== 可视化最佳分桶方案 ======
if res_df.empty:
    raise RuntimeError('未找到满足最小桶样本约束的方案，请降低 MIN_BUCKET_COUNT')

best = res_df.iloc[0].to_dict()
best_features = list(best['features'])
best_bins = int(best['n_bins'])

print('最佳方案:')
print(best)

work = df.copy()
for f in best_features:
    work[f'{f}_bucket_best'] = pd.qcut(work[f], q=best_bins, duplicates='drop')
work['best_bucket'] = work[[f'{f}_bucket_best' for f in best_features]].astype(str).agg(' | '.join, axis=1)

bucket_stats = work.groupby('best_bucket', observed=True)[ERROR_METRIC].agg(['mean', 'std', 'count']).sort_values('mean', ascending=False)
display(bucket_stats)

plt.figure(figsize=(max(10, min(22, len(bucket_stats) * 0.8)), 6))
order = bucket_stats.index.tolist()
sns.boxplot(data=work[work['best_bucket'].isin(order)], x='best_bucket', y=ERROR_METRIC, order=order, color='#4C72B0')
plt.xticks(rotation=45, ha='right')
plt.title(
    f"最佳分桶方案 | TSFM={TSFM_NAME} | 指标={ERROR_METRIC.upper()} | 特征={best_features} | bins={best_bins} | {scope_text}"
)
plt.xlabel('Best Buckets')
plt.ylabel(ERROR_METRIC.upper())
plt.tight_layout()
plt.show()
